In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]


Saving Hiba.csv to Hiba.csv


In [ ]:
from google.colab import files
uploaded = files.upload()
file_name2 = list(uploaded.keys())[0]


Saving PDS_dataset.csv to PDS_dataset.csv


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Define path to your uploaded zip file in Drive
zip_path = "/content/drive/MyDrive/skin_cancer_images_hiba.zip"   # change this to match your Drive

# Step 3: Unzip into a folder called 'images'
!unzip -q "$zip_path" -d images

# Step 4: Verify extraction
import os
print("Number of images extracted:", len(os.listdir("images")))
print("Sample files:", os.listdir("images")[:5])



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of images extracted: 1616
Sample files: ['ISIC_2608686.jpg', 'ISIC_9939005.jpg', 'ISIC_6652468.jpg', 'ISIC_8459589.jpg', 'ISIC_3553101.jpg']


In [ ]:
# from google.colab import drive
# import shutil
# import zipfile
# import os

# # Mount Drive
# drive.mount('/content/drive')

# # Paths to the existing zips
# zips = [
#     '/content/drive/MyDrive/imgs_part_1.zip',
#     '/content/drive/MyDrive/imgs_part_2.zip',
#     '/content/drive/MyDrive/imgs_part_3.zip'
# ]

# # Create a temporary extraction folder
# extract_folder = '/content/all_data'
# os.makedirs(extract_folder, exist_ok=True)

# # Extract all zip contents into one folder
# for z in zips:
#     with zipfile.ZipFile(z, 'r') as zip_ref:
#         zip_ref.extractall(extract_folder)

# # Now create one big zip of everything
# output_path = '/content/drive/MyDrive/combined_merged.zip'
# shutil.make_archive(output_path.replace('.zip',''), 'zip', extract_folder)

# print("✅ Merged zip created at:", output_path)


Mounted at /content/drive
✅ Merged zip created at: /content/drive/MyDrive/combined_merged.zip


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Define path to your uploaded zip file in Drive
zip_path = "/content/drive/MyDrive/combined_merged.zip"

# Step 3: Unzip into a folder called 'images'
!unzip -q "$zip_path" -d images2

# Step 4: Verify extraction
import os
print("Number of images extracted:", len(os.listdir("images2")))
print("Sample files:", os.listdir("images2")[:5])



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of images extracted: 3
Sample files: ['imgs_part_2', 'imgs_part_1', 'imgs_part_3']


In [ ]:
!find images2 -type f -name "*.jpg" -exec mv {} images2/ \;
!find images2 -type f -name "*.png" -exec mv {} images2/ \;


In [ ]:
import os
print("HIBA images:", len(os.listdir("images")))
print("PDS images:", len(os.listdir("images2")))
print("Example PDS files:", os.listdir("images2")[:5])


HIBA images: 1616
PDS images: 2301
Example PDS files: ['PAT_737_1394_308.png', 'PAT_106_158_249.png', 'PAT_992_1864_653.png', 'PAT_1492_1705_728.png', 'PAT_2008_4126_489.png']


In [ ]:
import os

image_count = 0
for root, dirs, files in os.walk("images2"):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
            image_count += 1

print("✅ Total number of images:", image_count)

✅ Total number of images: 2298


In [ ]:
!pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.5 MB/s eta 0:00:00


In [ ]:
"""
🎯 STABILIZED V4 FINAL - CONVERGENCE FIXES
=============================================================================
ALL Previous Optimizations + 6 Critical Stability Fixes
- Proper early stopping with checkpoint restore
- Conservative LR schedule (longer warmup, no aggressive restarts)
- Extended backbone freezing (stable domain adaptation)
- Stronger regularization (dropout + label smoothing + mixup)
- Optimized histogram matching (50 refs, less over-normalization)
- Simplified attention (SE blocks only, lightweight)
=============================================================================
"""

import os
import json
import pandas as pd
import numpy as np
from PIL import Image
import warnings
import gc
import cv2
import random
import pickle
from typing import Tuple, List, Dict, Optional
from skimage import io, exposure
from pathlib import Path

# Data processing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                            classification_report, confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
from torchvision.transforms import RandAugment, GaussianBlur, TrivialAugmentWide
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
import torch.nn.functional as F

# Utilities
from tqdm import tqdm
import joblib
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

# ==================== STABILIZED CONFIGURATION ====================

class StabilizedConfig:
    """Stabilized configuration with conservative settings"""
    # Fast debug mode
    FAST_DEBUG_MODE = False

    # Dataset paths
    HIBA_CSV = "Hiba.csv"
    PDS_CSV = "PDS_dataset.csv"
    HIBA_IMAGES_DIR = "images"
    PDS_IMAGES_DIR = "images2"

    # FIX 5: Reduced histogram references (less over-normalization)
    HISTOGRAM_REFERENCE_DIR = "images2"
    N_REFERENCE_IMAGES = 100  # FIX 5: Reduced from 200
    N_MEAN_COMPUTATION = 50
    USE_CACHED_HISTOGRAM = True
    USE_CLAHE_FALLBACK = True

    # Metadata caching
    METADATA_CACHE_PATH = "merged_metadata.csv"
    USE_CACHED_METADATA = True

    # Dataset control
    USE_HIBA = True
    USE_PDS = True
    COMBINE_DATASETS = True

    # Training hyperparameters
    SEED = 42
    N_SPLITS = 5 if not FAST_DEBUG_MODE else 1
    FUSION_EPOCHS = 60 if not FAST_DEBUG_MODE else 5
    BASE_LR = 1e-4  # FIX 2: More conservative

    # FIX 1: Proper early stopping
    USE_EARLY_STOPPING = True
    PATIENCE = 8  # FIX 1: Trigger after 8 epochs of no improvement
    EARLY_STOP_THRESHOLD = 0.01  # FIX 1: Min improvement threshold
    RESTORE_BEST_ON_STOP = True  # FIX 1: Load best checkpoint

    # FIX 3: Extended backbone freezing
    UNFREEZE_EPOCH = 12  # FIX 3: Later unfreezing (was 10)
    UNFREEZE_LR_FACTOR = 0.1

    # FIX 4: Stronger regularization
    DROPOUT_RATE = 0.5  # FIX 4: Increased from 0.4
    USE_LABEL_SMOOTHING = True
    LABEL_SMOOTHING_ALPHA = 0.15  # FIX 4: Increased from 0.1
    USE_MIXUP = True  # FIX 4: Added mixup
    MIXUP_ALPHA = 0.2
    MIN_CLASS_SAMPLES = 10

    # FIX 2: Conservative LR schedule
    USE_COSINE_ANNEALING = True  # FIX 2: Simple cosine, no restarts
    USE_REDUCE_ON_PLATEAU = True  # FIX 2: Additional safety net
    PLATEAU_PATIENCE = 5
    PLATEAU_FACTOR = 0.5

    # Class imbalance
    MAX_CLASS_WEIGHT_CAP = 100.0
    MINORITY_CLASS_THRESHOLD = 50
    USE_DYNAMIC_MINORITY_BOOST = True
    MIN_MINORITY_BOOST = 2.0
    MAX_MINORITY_BOOST = 4.0  # Reduced from 5.0
    POST_BOOST_CAP = 80.0  # Reduced from 100.0
    USE_LOG_SCALING_BOOST = True

    # Scoring
    F1W_WEIGHT = 0.5
    F1M_WEIGHT = 0.1
    AUC_WEIGHT = 0.4

    # Domain adaptation
    USE_HISTOGRAM_MATCHING = True
    USE_DOMAIN_NORMALIZATION = True
    USE_CROSS_DOMAIN_VALIDATION = True
    USE_DOMAIN_AWARE_FEATURES = True

    # Augmentation - Simplified
    USE_TRIVIAL_AUGMENT = True
    USE_GAUSSIAN_BLUR = True

    # CutMix disabled during stabilization
    USE_CUTMIX = False  # Disabled for stability

    # Logging
    LOG_PER_CLASS_METRICS = True
    SAVE_CONFUSION_MATRIX = True
    LOG_INTERVAL = 5  # More frequent

    # Mixed precision
    USE_MIXED_PRECISION = True

    # Checkpoint management
    CHECKPOINT_DIR = "checkpoints_stabilized"
    SAVE_FULL_MODEL = True

    # FIX 6: Simplified attention
    USE_SE_ONLY = True  # FIX 6: SE blocks only, no CBAM

    # Device
    DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    BATCH_SIZE = 32 if torch.cuda.is_available() else 8
    if FAST_DEBUG_MODE:
        BATCH_SIZE = 8
    NUM_WORKERS = min(8, os.cpu_count() // 2) if torch.cuda.is_available() else 2

config = StabilizedConfig()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(config.SEED)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

writer = SummaryWriter(log_dir="runs/stabilized_v4")

print(f"✅ Device: {config.DEVICE}")
print(f"✅ Stabilized V4: 6 critical stability fixes")
print(f"   - Early stopping with restore")
print(f"   - Conservative LR (1e-4, cosine)")
print(f"   - Extended freezing (12 epochs)")
print(f"   - Strong regularization (dropout=0.5, LS=0.15, mixup)")
print(f"   - Optimized histogram (50 refs)")
print(f"   - Simplified attention (SE only)")

# ==================== DIAGNOSIS MAPPING ====================

DIAGNOSIS_MAPPING = {
    'nevus': 'NEV', 'basal cell carcinoma': 'BCC', 'melanoma': 'MEL',
    'actinic keratosis': 'ACK', 'seborrheic keratosis': 'SEK',
    'squamous cell carcinoma': 'SCC', 'dermatofibroma': 'DF',
    'vascular lesion': 'VASC', 'lichenoid keratosis': 'LK',
    'NEV': 'NEV', 'BCC': 'BCC', 'MEL': 'MEL', 'ACK': 'ACK',
    'SEK': 'SEK', 'SCC': 'SCC', 'DF': 'DF', 'VASC': 'VASC', 'LK': 'LK'
}

FITZPATRICK_MAPPING = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6,
    '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6,
    1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6,
    1.0: 1, 2.0: 2, 3.0: 3, 4.0: 4, 5.0: 5, 6.0: 6
}

def convert_fitzpatrick(value) -> int:
    if pd.isna(value):
        return 3
    val_str = str(value).strip().upper()
    return FITZPATRICK_MAPPING.get(val_str, 3)

# ==================== OPTIMIZED HISTOGRAM MATCHING ====================

def gather_reference_paths(ref_dir: str, n_refs: int = 50, seed: int = 42) -> List[str]:
    all_files = []

    if not os.path.exists(ref_dir):
        return []

    for root, _, files in os.walk(ref_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_files.append(os.path.join(root, f))

    random.seed(seed)
    if n_refs is None or n_refs >= len(all_files):
        return all_files
    return random.sample(all_files, min(n_refs, len(all_files)))

def compute_mean_reference_image(image_paths: List[str], n_compute: int = 100) -> Optional[np.ndarray]:
    if not image_paths:
        return None

    images = []
    for p in image_paths[:n_compute]:
        try:
            img = io.imread(p)

            # 🔧 Ensure image has 3 channels
            if img.ndim == 2:
                img = np.stack([img] * 3, axis=-1)
            elif img.shape[-1] > 3:
                img = img[..., :3]

            # 🔧 Handle invalid or empty images
            if img.size == 0:
                continue

            # 🔧 Normalize dtype
            if img.dtype != np.uint8:
                img = (255 * (img / img.max())).astype(np.uint8)

            # 🔧 Enforce consistent size
            img = cv2.resize(img, (224, 224))

            images.append(img.astype(np.float32))
        except Exception as e:
            # Optionally print errors for debugging
            # print(f"⚠️ Skipped {p}: {e}")
            continue

    if images:
        # 🔧 Stack safely before averaging to avoid shape mismatch
        images = np.stack(images, axis=0)
        mean_img = np.mean(images, axis=0).astype(np.uint8)
        return mean_img

    return None


class OptimizedHistogramMatcher:
    def __init__(self, reference_dir: str):
        self.reference_image = None
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        self.use_clahe = config.USE_CLAHE_FALLBACK
        self._load_reference(reference_dir)

    def _load_reference(self, ref_dir: str):
        print(f"🎨 Loading histogram reference (50 images)...")

        cache_path = f"histogram_{os.path.basename(ref_dir)}_50.pkl"

        if config.USE_CACHED_HISTOGRAM and os.path.exists(cache_path):
            try:
                with open(cache_path, 'rb') as f:
                    self.reference_image = pickle.load(f)
                print(f"   ✅ Loaded cached reference")
                return
            except:
                pass

        ref_paths = gather_reference_paths(ref_dir, n_refs=config.N_REFERENCE_IMAGES, seed=config.SEED)

        if not ref_paths:
            print(f"   ⚠️ No references found")
            return

        self.reference_image = compute_mean_reference_image(ref_paths, n_compute=config.N_MEAN_COMPUTATION)

        if self.reference_image is not None:
            try:
                with open(cache_path, 'wb') as f:
                    pickle.dump(self.reference_image, f)
                print(f"   ✅ Cached reference ({len(ref_paths)} samples)")
            except:
                pass
        else:
            self.use_clahe = True

    def match_histogram(self, img_np: np.ndarray) -> np.ndarray:
        if self.reference_image is None or self.use_clahe:
            try:
                if img_np.ndim == 3:
                    channels = cv2.split(img_np)
                    channels = [self.clahe.apply(ch) for ch in channels]
                    return cv2.merge(channels)
                else:
                    return self.clahe.apply(img_np)
            except:
                return img_np

        try:
            ref_resized = cv2.resize(self.reference_image, (img_np.shape[1], img_np.shape[0]))
            matched = exposure.match_histograms(img_np, ref_resized, channel_axis=-1)
            return matched.astype(np.uint8)
        except:
            return img_np

# ==================== ENHANCED AUGMENTATION ====================

def get_stabilized_training_augmentation():
    """Stabilized augmentation with mixup"""
    return transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(15),  # Reduced from 20
        TrivialAugmentWide(),
        GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def get_minority_augmentation():
    return transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(0.7),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(30),
        RandAugment(num_ops=3, magnitude=10),
        GaussianBlur(kernel_size=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def get_val_augmentation():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

# ==================== LOSS FUNCTIONS ====================

class LabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes: int, alpha: float = 0.15):
        super().__init__()
        self.num_classes = num_classes
        self.alpha = alpha
        self.criterion = nn.KLDivLoss(reduction='batchmean')

    def forward(self, outputs, targets):
        log_probs = F.log_softmax(outputs, dim=-1)
        smooth_labels = torch.full_like(log_probs, self.alpha / (self.num_classes - 1))
        smooth_labels.scatter_(1, targets.unsqueeze(1), 1 - self.alpha)
        return self.criterion(log_probs, smooth_labels)

def mixup_data(x, y, alpha=0.2):
    """FIX 4: Mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ==================== SIMPLIFIED MODEL ====================

class SEBlock(nn.Module):
    """FIX 6: SE blocks only"""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, _, _ = x.size()
        y = F.adaptive_avg_pool2d(x, 1).view(b, c)
        y = torch.sigmoid(self.fc2(F.relu(self.fc1(y))))
        return x * y.view(b, c, 1, 1)

class BalancedGatedFusion(nn.Module):
    def __init__(self, cnn_dim: int, ann_dim: int):
        super().__init__()
        self.ann_proj = nn.Linear(ann_dim, cnn_dim)
        self.gate = nn.Sequential(
            nn.Linear(cnn_dim, cnn_dim // 4),
            nn.ReLU(),
            nn.Linear(cnn_dim // 4, cnn_dim),
            nn.Sigmoid()
        )

    def forward(self, cnn_features, ann_features):
        ann_proj = self.ann_proj(ann_features)
        g = self.gate(cnn_features)
        return g * cnn_features + (1 - g) * ann_proj

class StabilizedModel(nn.Module):
    """Stabilized model with SE only"""
    def __init__(self, input_dim: int, num_classes: int, dropout_rate: float = 0.5):
        super().__init__()

        # CNN branch
        self.backbone = efficientnet_b1(weights=EfficientNet_B1_Weights.IMAGENET1K_V1)
        self.features = self.backbone.features
        img_feat_dim = 1280

        # FIX 3: Initially frozen
        for param in self.features.parameters():
            param.requires_grad = False

        self.global_avg = nn.AdaptiveAvgPool2d(1)
        self.global_max = nn.AdaptiveMaxPool2d(1)

        # FIX 6: SE blocks only
        self.se_block = SEBlock(img_feat_dim)

        # FIX 4: Stronger dropout
        self.img_projection = nn.Sequential(
            nn.Linear(img_feat_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(dropout_rate * 0.5)  # Additional dropout
        )

        # Tabular branch
        tab_input_dim = input_dim + 1 if config.USE_DOMAIN_AWARE_FEATURES else input_dim

        self.tab_processor = nn.Sequential(
            nn.BatchNorm1d(tab_input_dim),
            nn.Linear(tab_input_dim, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(dropout_rate * 0.5)
        )

        # Gated fusion
        self.gated_fusion = BalancedGatedFusion(256, 128)

        # FIX 4: Stronger final dropout
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )

    def forward(self, img, features):
        x_img = self.features(img)
        x_img = self.se_block(x_img)
        avg = self.global_avg(x_img).view(img.size(0), -1)
        max_p = self.global_max(x_img).view(img.size(0), -1)
        x_img = torch.cat([avg, max_p], dim=1)
        x_img = self.img_projection(x_img)

        x_tab = self.tab_processor(features)

        x_fused = self.gated_fusion(x_img, x_tab)

        return self.classifier(x_fused)

# ==================== DATASET ====================

class OptimizedMultiModalDataset(Dataset):
    def __init__(self, df, image_dirs, transform, minority_transform, tab_features=None,
                 histogram_matcher=None, add_domain_feature=False, class_counts=None):
        self.df = df.reset_index(drop=True)
        self.image_dirs = image_dirs
        self.transform = transform
        self.minority_transform = minority_transform
        self.tab_features = tab_features
        self.histogram_matcher = histogram_matcher
        self.add_domain_feature = add_domain_feature

        self.image_ids = df['image_id'].values
        self.labels = torch.tensor(df['target'].values, dtype=torch.long)
        self.domains = torch.tensor(
            (df['dataset'].map({'HIBA': 0, 'PDS': 1}).fillna(0)).values,
            dtype=torch.float32
        )

        if class_counts:
            self.minority_classes = set([cls for cls, count in class_counts.items()
                                       if count < config.MINORITY_CLASS_THRESHOLD])
        else:
            self.minority_classes = set()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        label = self.labels[idx]
        domain = self.domains[idx]

        img_path = None
        for img_dir in self.image_dirs:
            for ext in ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'):
                p = os.path.join(img_dir, f"{img_id}{ext}")
                if os.path.exists(p):
                    img_path = p
                    break
            if img_path:
                break

        if img_path:
            try:
                img = Image.open(img_path).convert('RGB')
                img_np = np.array(img)

                if self.histogram_matcher is not None:
                    img_np = self.histogram_matcher.match_histogram(img_np)
                    img = Image.fromarray(img_np)
            except:
                img = Image.new('RGB', (224, 224), color=(128, 128, 128))
        else:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))

        if (int(label.item()) in self.minority_classes and
            self.minority_transform is not None):
            img = self.minority_transform(img)
        else:
            if self.transform:
                img = self.transform(img)

        features = torch.tensor(self.tab_features[idx], dtype=torch.float32) if self.tab_features is not None else torch.zeros(0)

        if self.add_domain_feature:
            features = torch.cat([features, domain.unsqueeze(0)])

        return img, features, label

# ==================== PREPROCESSING ====================

class CachedDomainAwarePreprocessor:
    def __init__(self):
        self.numeric_cols = ['age']
        self.ordinal_cols = ['fitzpatrick']
        self.categorical_cols = ['gender', 'anatomical_site']
        self.domain_scalers = {}
        self.fitted = False

    def fit_and_transform(self, train_df, val_df):
        train_df = train_df.copy()
        val_df = val_df.copy()

        for col in self.numeric_cols:
            if col in train_df.columns:
                for domain in train_df['dataset'].unique():
                    domain_train = train_df[train_df['dataset'] == domain]
                    if len(domain_train) > 1:
                        scaler = StandardScaler()
                        train_df.loc[train_df['dataset'] == domain, col] = \
                            scaler.fit_transform(domain_train[[col]])
                        self.domain_scalers[f"{col}_{domain}"] = scaler

                        domain_val = val_df[val_df['dataset'] == domain]
                        if len(domain_val) > 0:
                            val_df.loc[val_df['dataset'] == domain, col] = \
                                scaler.transform(domain_val[[col]])

        for col in self.ordinal_cols:
            if col in train_df.columns:
                scaler = StandardScaler()
                train_df[col] = scaler.fit_transform(train_df[[col]])
                val_df[col] = scaler.transform(val_df[[col]])

        for col in self.categorical_cols:
            if col in train_df.columns:
                encoder = LabelEncoder()
                train_vals = train_df[col].fillna('unknown').astype(str)
                val_vals = val_df[col].fillna('unknown').astype(str)
                encoder.fit(train_vals)
                train_df[col] = encoder.transform(train_vals)
                val_encoded = []
                for v in val_vals:
                    try:
                        val_encoded.append(encoder.transform([v])[0])
                    except:
                        val_encoded.append(0)
                val_df[col] = val_encoded

        self.fitted = True
        return train_df, val_df

    def prepare_features(self, df) -> np.ndarray:
        feature_cols = ([c for c in self.numeric_cols if c in df.columns] +
                       [c for c in self.ordinal_cols if c in df.columns] +
                       [c for c in self.categorical_cols if c in df.columns])
        df_feat = df[feature_cols].copy()
        for col in df_feat.columns:
            df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce').fillna(0)
        return df_feat.values.astype(np.float32)

# ==================== UTILITIES ====================

def safe_auc(y_true: np.ndarray, y_prob: np.ndarray, num_classes: int) -> float:
    try:
        if len(np.unique(y_true)) < 2:
            return 0.0
        y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
        if y_true_bin.ndim == 1:
            y_true_bin = y_true_bin.reshape(-1, 1)
        if y_true_bin.shape[1] == 1:
            return 0.0

        y_prob_clipped = y_prob[:, :num_classes]
        return roc_auc_score(y_true_bin, y_prob_clipped,
                           average='weighted', multi_class='ovr')
    except:
        return 0.0

def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def log_per_class_metrics(y_true, y_pred, y_prob, class_names, fold, epoch):
    if not config.LOG_PER_CLASS_METRICS:
        return

    report = classification_report(y_true, y_pred, target_names=class_names,
                                 digits=3, zero_division=0)

    os.makedirs("reports", exist_ok=True)
    with open(f"reports/fold_{fold}_epoch_{epoch}_report.txt", "w") as f:
        f.write(report)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    print(f"\n📊 Per-Class (E{epoch}):")
    print(f"{'Class':<10} {'P':<7} {'R':<7} {'F1':<7} {'S':<7}")
    for i, class_name in enumerate(class_names):
        print(f"{class_name:<10} {precision[i]:>6.3f} {recall[i]:>6.3f} {f1[i]:>6.3f} {int(support[i]):>6}")

    if config.SAVE_CONFUSION_MATRIX:
        cm = confusion_matrix(y_true, y_pred)
        np.savetxt(f"reports/fold_{fold}_epoch_{epoch}_confusion.csv", cm,
                  delimiter=",", fmt='%d')

def compute_dynamic_boost(class_size: int, total_size: int) -> float:
    if config.USE_LOG_SCALING_BOOST:
        ratio = total_size / max(class_size, 1)
        boost = config.MIN_MINORITY_BOOST + config.MAX_MINORITY_BOOST * np.log1p(ratio) / np.log1p(total_size)
    else:
        boost = config.MAX_MINORITY_BOOST * (total_size / max(class_size, 1)) / 100

    return min(boost, config.MAX_MINORITY_BOOST)

# ==================== FIX 1: EARLY STOPPING CLASS ====================

class EarlyStopping:
    """FIX 1: Proper early stopping with checkpoint restore"""
    def __init__(self, patience=8, threshold=0.01, mode='max'):
        self.patience = patience
        self.threshold = threshold
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.should_stop = False
        self.best_epoch = 0

    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False

        if self.mode == 'max':
            if score > self.best_score + self.threshold:
                self.best_score = score
                self.best_epoch = epoch
                self.counter = 0
                return False
            else:
                self.counter += 1
        else:
            if score < self.best_score - self.threshold:
                self.best_score = score
                self.best_epoch = epoch
                self.counter = 0
                return False
            else:
                self.counter += 1

        if self.counter >= self.patience:
            self.should_stop = True
            return True

        return False

# ==================== MAIN TRAINING ====================

def train_stabilized(fold: int, train_idx: np.ndarray, val_idx: np.ndarray,
                    df: pd.DataFrame, image_dirs: List[str], num_classes: int,
                    class_names: List[str], histogram_matcher=None) -> Dict:
    """Stabilized training with all fixes"""
    print(f"\n{'='*60}\nFOLD {fold} - STABILIZED\n{'='*60}")

    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    class_counts = dict(train_df['target'].value_counts())

    class_weights = compute_class_weight('balanced', classes=np.arange(num_classes),
                                        y=train_df['target'].values)
    class_weights = np.clip(class_weights, 0.1, config.MAX_CLASS_WEIGHT_CAP)

    preprocessor = CachedDomainAwarePreprocessor()
    train_df, val_df = preprocessor.fit_and_transform(train_df, val_df)

    train_features = preprocessor.prepare_features(train_df)
    val_features = preprocessor.prepare_features(val_df)

    train_ds = OptimizedMultiModalDataset(
        train_df, image_dirs, get_stabilized_training_augmentation(), get_minority_augmentation(),
        train_features, histogram_matcher,
        add_domain_feature=config.USE_DOMAIN_AWARE_FEATURES,
        class_counts=class_counts
    )
    val_ds = OptimizedMultiModalDataset(
        val_df, image_dirs, get_val_augmentation(), None,
        val_features, histogram_matcher,
        add_domain_feature=config.USE_DOMAIN_AWARE_FEATURES
    )

    minority_classes = set([cls for cls, count in class_counts.items()
                          if count < config.MINORITY_CLASS_THRESHOLD])

    sample_weights = []
    for i, row in train_df.iterrows():
        class_weight = class_weights[row['target']]
        if row['target'] in minority_classes:
            boost = compute_dynamic_boost(class_counts[row['target']], len(train_df))
            class_weight *= boost

        class_weight = min(class_weight, config.POST_BOOST_CAP)
        sample_weights.append(class_weight)

    sample_weights = torch.tensor(sample_weights, dtype=torch.float)
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, sampler=sampler,
                             num_workers=config.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE*2, shuffle=False,
                           num_workers=config.NUM_WORKERS, pin_memory=True)

    model = StabilizedModel(train_features.shape[1], num_classes, config.DROPOUT_RATE)
    model = model.to(config.DEVICE)

    # FIX 4: Label smoothing criterion
    criterion = LabelSmoothingLoss(num_classes, alpha=config.LABEL_SMOOTHING_ALPHA).to(config.DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=config.BASE_LR, weight_decay=1e-4)

    # FIX 2: Conservative LR schedule
    if config.USE_COSINE_ANNEALING:
        scheduler_cosine = CosineAnnealingLR(optimizer, T_max=config.FUSION_EPOCHS, eta_min=1e-6)

    if config.USE_REDUCE_ON_PLATEAU:
        scheduler_plateau = ReduceLROnPlateau(optimizer, mode='max', factor=config.PLATEAU_FACTOR,
                                            patience=config.PLATEAU_PATIENCE)

    scaler = GradScaler() if config.USE_MIXED_PRECISION else None

    # FIX 1: Early stopping
    early_stopping = EarlyStopping(patience=config.PATIENCE, threshold=config.EARLY_STOP_THRESHOLD, mode='max')

    best_f1 = 0.0
    best_auc = 0.0
    best_f1m = 0.0
    best_composite = -1.0
    backbone_unfrozen = False

    fold_checkpoint_dir = os.path.join(config.CHECKPOINT_DIR, f"fold_{fold}")
    os.makedirs(fold_checkpoint_dir, exist_ok=True)
    checkpoint_path = os.path.join(fold_checkpoint_dir, "best_model.pth")

    for epoch in range(1, config.FUSION_EPOCHS + 1):
        # FIX 3: Extended freezing
        if epoch >= config.UNFREEZE_EPOCH and not backbone_unfrozen:
            print(f"🔓 Unfreezing backbone at epoch {epoch}...")
            for param in model.features.parameters():
                param.requires_grad = True
            for param_group in optimizer.param_groups:
                param_group['lr'] *= config.UNFREEZE_LR_FACTOR
            backbone_unfrozen = True

        model.train()
        train_losses = []

        for batch in tqdm(train_loader, desc=f"E{epoch}", leave=False):
            imgs, feats, lbls = batch
            imgs, feats, lbls = imgs.to(config.DEVICE), feats.to(config.DEVICE), lbls.to(config.DEVICE)

            optimizer.zero_grad(set_to_none=True)

            # FIX 4: Mixup
            if config.USE_MIXUP and epoch >= 5:  # Start mixup after epoch 5
                imgs, lbls_a, lbls_b, lam = mixup_data(imgs, lbls, alpha=config.MIXUP_ALPHA)

                if config.USE_MIXED_PRECISION:
                    with autocast():
                        outputs = model(imgs, feats)
                        loss = mixup_criterion(criterion, outputs, lbls_a, lbls_b, lam)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(imgs, feats)
                    loss = mixup_criterion(criterion, outputs, lbls_a, lbls_b, lam)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
            else:
                if config.USE_MIXED_PRECISION:
                    with autocast():
                        outputs = model(imgs, feats)
                        loss = criterion(outputs, lbls)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(imgs, feats)
                    loss = criterion(outputs, lbls)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            train_losses.append(loss.item())

        # FIX 2: Update schedulers
        if config.USE_COSINE_ANNEALING:
            scheduler_cosine.step()

        # Validation
        model.eval()
        val_losses, val_preds, val_probs, val_labels = [], [], [], []

        with torch.no_grad():
            for batch in val_loader:
                imgs, feats, lbls = batch
                imgs, feats, lbls = imgs.to(config.DEVICE), feats.to(config.DEVICE), lbls.to(config.DEVICE)

                if config.USE_MIXED_PRECISION:
                    with autocast():
                        outputs = model(imgs, feats)
                        loss = criterion(outputs, lbls)
                else:
                    outputs = model(imgs, feats)
                    loss = criterion(outputs, lbls)

                val_losses.append(loss.item())
                val_probs.extend(F.softmax(outputs, dim=1).cpu().numpy())
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(lbls.cpu().numpy())

        val_acc = accuracy_score(val_labels, val_preds)
        val_f1_w = f1_score(val_labels, val_preds, average='weighted', zero_division=0)
        val_f1_m = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        val_auc = safe_auc(np.array(val_labels), np.array(val_probs), num_classes)

        composite = (config.F1W_WEIGHT * val_f1_w +
                    config.F1M_WEIGHT * val_f1_m +
                    config.AUC_WEIGHT * (val_auc if val_auc > 0 else 0))

        # FIX 2: Update plateau scheduler
        if config.USE_REDUCE_ON_PLATEAU:
            scheduler_plateau.step(composite)

        if epoch % config.LOG_INTERVAL == 0 or composite > best_composite:
            log_per_class_metrics(val_labels, val_preds, np.array(val_probs), class_names, fold, epoch)

        print(f"E{epoch:2d} Loss:{np.mean(train_losses):.3f}/{np.mean(val_losses):.3f} "
              f"Acc:{val_acc:.3f} F1w:{val_f1_w:.3f} F1m:{val_f1_m:.3f} AUC:{val_auc:.3f} C:{composite:.3f}")

        # Save best
        if composite > best_composite:
            best_composite = composite
            best_f1 = val_f1_w
            best_f1m = val_f1_m
            best_auc = val_auc

            torch.save(model.state_dict(), checkpoint_path)
            if config.SAVE_FULL_MODEL:
                full_model_path = os.path.join(fold_checkpoint_dir, "best_model_full.pt")
                torch.save(model, full_model_path)
            print(f"   ✅ Best model saved (composite={composite:.3f})")

        # FIX 1: Early stopping
        if config.USE_EARLY_STOPPING:
            if early_stopping(composite, epoch):
                print(f"\n⏹️ Early stop at epoch {epoch} (best: E{early_stopping.best_epoch}, score={early_stopping.best_score:.3f})")

                # FIX 1: Restore best checkpoint
                if config.RESTORE_BEST_ON_STOP:
                    print(f"   ↩️ Restoring best checkpoint from epoch {early_stopping.best_epoch}")
                    model.load_state_dict(torch.load(checkpoint_path))
                break

        if epoch % 10 == 0:
            aggressive_cleanup()

    return {
        'fold': fold, 'best_f1': best_f1, 'best_f1m': best_f1m, 'best_auc': best_auc,
        'composite': best_composite
    }

# ==================== DATA LOADING ====================

class DatasetHarmonizer:
    def load_hiba(self) -> pd.DataFrame:
        print("\n📊 Loading HIBA...")
        df = pd.read_csv(config.HIBA_CSV)
        df['image_id'] = df.get('isic_id', df.get('image_id', ''))
        df['gender'] = df.get('sex', 'unknown').str.lower()
        df['age'] = pd.to_numeric(df.get('age_approx', 0), errors='coerce').fillna(30).astype(int)
        df['fitzpatrick'] = df.get('fitzpatrick_skin_type', 3).apply(convert_fitzpatrick)
        df['anatomical_site'] = df.get('anatom_site_general', 'unknown').str.lower()
        df['diagnosis'] = df.get('diagnosis', '').str.lower().map(DIAGNOSIS_MAPPING)
        df['dataset'] = 'HIBA'
        if 'patient_id' not in df.columns:
            df['patient_id'] = df['image_id']
        print(f"   ✅ {len(df)}")
        return df

    def load_pds(self) -> pd.DataFrame:
        print("\n📊 Loading PDS...")
        df = pd.read_csv(config.PDS_CSV)
        df['image_id'] = df.get('img_id', '')
        df['gender'] = df.get('gender', 'unknown').str.lower()
        df['age'] = pd.to_numeric(df.get('age', 0), errors='coerce').fillna(30).astype(int)
        df['fitzpatrick'] = df.get('fitspatrick', 3).apply(convert_fitzpatrick)
        df['anatomical_site'] = df.get('region', 'unknown').str.lower()
        df['diagnosis'] = df.get('diagnostic', '').str.upper().map(DIAGNOSIS_MAPPING)
        df['dataset'] = 'PDS'
        if 'patient_id' not in df.columns:
            df['patient_id'] = df.get('patient_id', df['image_id'])
        print(f"   ✅ {len(df)}")
        return df

    def merge(self, df_hiba: pd.DataFrame, df_pds: pd.DataFrame) -> pd.DataFrame:
        print("\n🔀 Merging...")
        cols = ['image_id', 'gender', 'age', 'fitzpatrick', 'anatomical_site',
                'diagnosis', 'patient_id', 'dataset']
        for col in cols:
            for df in [df_hiba, df_pds]:
                if col not in df.columns:
                    df[col] = 'unknown' if col in ['gender', 'anatomical_site', 'diagnosis', 'dataset'] else 0

        df_merged = pd.concat([df_hiba[cols], df_pds[cols]], ignore_index=True)
        print(f"   ✅ {len(df_merged)}")
        return df_merged

# ==================== MAIN ====================

print("="*60)
print("🎯 STABILIZED V4 - 6 CRITICAL FIXES")
print("="*60)

harmonizer = DatasetHarmonizer()

if config.USE_CACHED_METADATA and os.path.exists(config.METADATA_CACHE_PATH):
    print("\n📦 Loading cached metadata...")
    df_full = pd.read_csv(config.METADATA_CACHE_PATH)
    print(f"   ✅ {len(df_full)}")
else:
    df_hiba = harmonizer.load_hiba() if config.USE_HIBA else pd.DataFrame()
    df_pds = harmonizer.load_pds() if config.USE_PDS else pd.DataFrame()

    if config.COMBINE_DATASETS and len(df_hiba) > 0 and len(df_pds) > 0:
        df_full = harmonizer.merge(df_hiba, df_pds)
    elif len(df_hiba) > 0:
        df_full = df_hiba
    else:
        df_full = df_pds

    if config.USE_CACHED_METADATA:
        df_full.to_csv(config.METADATA_CACHE_PATH, index=False)

print(f"\n📊 Filtering...")
class_counts = df_full['diagnosis'].value_counts()
rare_classes = class_counts[class_counts < config.MIN_CLASS_SAMPLES].index
df_full = df_full[~df_full['diagnosis'].isin(rare_classes)].reset_index(drop=True)
df_full = df_full.dropna(subset=['diagnosis']).reset_index(drop=True)

encoder = LabelEncoder()
df_full['target'] = encoder.fit_transform(df_full['diagnosis'])
num_classes = len(encoder.classes_)
class_names = list(encoder.classes_)

print(f"📊 Classes: {class_names}, Total: {len(df_full)}")

image_dirs = []
if config.USE_HIBA:
    image_dirs.append(config.HIBA_IMAGES_DIR)
if config.USE_PDS:
    image_dirs.append(config.PDS_IMAGES_DIR)

histogram_matcher = None
if config.USE_HISTOGRAM_MATCHING:
    histogram_matcher = OptimizedHistogramMatcher(config.HISTOGRAM_REFERENCE_DIR)

print("\n" + "="*60)
print("🚀 Starting Stabilized Training")
print("="*60)

skf = StratifiedKFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.SEED)
all_results = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full['target']), 1):
    result = train_stabilized(fold_idx, train_idx, val_idx, df_full,
                             image_dirs, num_classes, class_names, histogram_matcher)
    all_results.append(result)
    aggressive_cleanup()

print("\n" + "="*60)
print("🏆 STABILIZED V4 FINAL RESULTS")
print("="*60)

results_df = pd.DataFrame(all_results)
results_df.to_csv("results_stabilized_v4.csv", index=False)

print(f"\n📊 Performance:")
print(f"   F1w: {results_df['best_f1'].mean():.4f} ± {results_df['best_f1'].std():.4f}")
print(f"   F1m: {results_df['best_f1m'].mean():.4f} ± {results_df['best_f1m'].std():.4f}")
print(f"   AUC: {results_df['best_auc'].mean():.4f} ± {results_df['best_auc'].std():.4f}")
print(f"   Composite: {results_df['composite'].mean():.4f} ± {results_df['composite'].std():.4f}")

writer.close()
print(f"\n✅ Stabilized training complete!")
print(f"📁 Results: results_stabilized_v4.csv")
print(f"📁 Checkpoints: {config.CHECKPOINT_DIR}/")

✅ Device: cuda:0
✅ Stabilized V4: 6 critical stability fixes
   - Early stopping with restore
   - Conservative LR (1e-4, cosine)
   - Extended freezing (12 epochs)
   - Strong regularization (dropout=0.5, LS=0.15, mixup)
   - Optimized histogram (50 refs)
   - Simplified attention (SE only)
🎯 STABILIZED V4 - 6 CRITICAL FIXES

📊 Loading HIBA...
   ✅ 1616

📊 Loading PDS...
   ✅ 2298

🔀 Merging...
   ✅ 3914

📊 Filtering...
📊 Classes: ['ACK', 'BCC', 'DF', 'MEL', 'NEV', 'SCC', 'SEK', 'VASC'], Total: 3894
🎨 Loading histogram reference (50 images)...
   ✅ Cached reference (100 samples)

🚀 Starting Stabilized Training

FOLD 1 - STABILIZED
Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth


100%|██████████| 30.1M/30.1M [00:00<00:00, 169MB/s]



📊 Per-Class (E1):
Class      P       R       F1      S      
ACK         0.198  0.113  0.144    159
BCC         0.657  0.186  0.289    237
DF          0.064  1.000  0.121     12
MEL         0.333  0.049  0.086     61
NEV         0.704  0.112  0.193    170
SCC         0.213  0.286  0.244     70
SEK         0.196  0.650  0.301     60
VASC        0.057  0.600  0.104     10
E 1 Loss:1.166/1.158 Acc:0.207 F1w:0.215 F1m:0.185 AUC:0.797 C:0.444
   ✅ Best model saved (composite=0.444)


E 2 Loss:0.881/1.119 Acc:0.243 F1w:0.175 F1m:0.155 AUC:0.807 C:0.426



📊 Per-Class (E3):
Class      P       R       F1      S      
ACK         0.432  0.736  0.544    159
BCC         0.623  0.363  0.459    237
DF          0.069  0.917  0.128     12
MEL         0.389  0.115  0.177     61
NEV         0.792  0.247  0.377    170
SCC         0.282  0.157  0.202     70
SEK         0.250  0.033  0.059     60
VASC        0.065  0.600  0.118     10
E 3 Loss:0.771/1.044 Acc:0.362 F1w:0.373 F1m:0.258 AUC:0.819 C:0.540
   ✅ Best model saved (composite=0.540)


...

E19 Loss:0.774/0.943 Acc:0.499 F1w:0.520 F1m:0.406 AUC:0.824 C:0.630



📊 Per-Class (E20):
Class      P       R       F1      S      
ACK         0.562  0.516  0.538    159
BCC         0.570  0.549  0.559    237
DF          0.167  0.583  0.259     12
MEL         0.329  0.377  0.351     61
NEV         0.842  0.663  0.742    169
SCC         0.308  0.343  0.324     70
SEK         0.346  0.150  0.209     60
VASC        0.125  0.636  0.209     11
E20 Loss:0.745/0.930 Acc:0.506 F1w:0.520 F1m:0.399 AUC:0.824 C:0.630


E21 Loss:0.740/0.932 Acc:0.510 F1w:0.526 F1m:0.409 AUC:0.824 C:0.633



📊 Per-Class (E22):
Class      P       R       F1      S      
ACK         0.652  0.472  0.547    159
BCC         0.584  0.603  0.593    237
DF          0.151  0.667  0.246     12
MEL         0.377  0.426  0.400     61
NEV         0.844  0.675  0.750    169
SCC         0.346  0.386  0.365     70
SEK         0.343  0.200  0.253     60
VASC        0.143  0.636  0.233     11
E22 Loss:0.755/0.927 Acc:0.529 F1w:0.546 F1m:0.423 AUC:0.822 C:0.644
   ✅ Best model saved (composite=0.644)


E23:  14%|█▍        | 14/98 [00:55<05:44,  4.10s/it]